<a href="https://colab.research.google.com/github/Ankit0974/Mental_Health_Detector/blob/main/HealthModelSave_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q -U transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 112.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab_Notebooks/Suicide_Detection_Shuffled.csv')


In [ ]:
df.head()

,text,class
0,im gonna sleep so put things in the comments f...,non-suicide
1,There is nothing for me to look forward to.Edi...,suicide
2,Anyone else get their education fucked by Coro...,non-suicide
3,Lemon tea &amp; weezer Filler filler filler fi...,non-suicide
4,"sorry for the annoyancefriend 1 is ""tired of m...",suicide


In [ ]:
df = df[['text', 'class']]
df['class'] = df['class'].str.strip().str.lower()

df['label'] = df['class'].map({
    'suicide': 1,
    'non-suicide': 0
})

df = df[['text', 'label']]


In [ ]:
!pip uninstall -y scikit-learn
!pip install -U scikit-learn

Found existing installation: scikit-learn 1.9.0
Uninstalling scikit-learn-1.9.0:
  Successfully uninstalled scikit-learn-1.9.0
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.1 MB)


In [ ]:
# import sys
# !{sys.executable} -m pip install scikit-learn==1.2.2
from sklearn.model_selection import train_test_split

# 80% Train, 20% Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    df["text"],
    df["label"],
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

# Split Temp into Validation (10%) and Test (10%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(len(X_train), len(X_val), len(X_test))

186568 23321 23321


In [ ]:
from datasets import Dataset
import pandas as pd

train_df = pd.DataFrame({
    "text": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "text": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "text": X_test,
    "label": y_test
})

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/186568 [00:00<?, ? examples/s]

Map:   0%|          | 0/23321 [00:00<?, ? examples/s]

Map:   0%|          | 0/23321 [00:00<?, ? examples/s]

In [ ]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

In [ ]:
import transformers
print(transformers.__version__)


5.12.1


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=4,

    weight_decay=0.01,

    fp16=True,

    logging_steps=200,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.108929,0.096440,0.973543,0.969089,0.978025,0.973536
2,0.078179,0.096097,0.975301,0.976495,0.973802,0.975147
3,0.049938,0.137875,0.974701,0.966774,0.982937,0.974788
4,0.017384,0.133831,0.974358,0.967148,0.981817,0.974427


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=46644, training_loss=0.06729599061659174, metrics={'train_runtime': 5508.1997, 'train_samples_per_second': 135.484, 'train_steps_per_second': 8.468, 'total_flos': 4.908810337640448e+16, 'train_loss': 0.06729599061659174, 'epoch': 4.0})

In [ ]:
import transformers
import datasets
import torch
import torchvision

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

Transformers: 5.13.1
Datasets: 5.0.0
Torch: 2.11.0+cu128
Torchvision: 0.26.0+cu128


In [ ]:
model.save_pretrained("/content/drive/MyDrive/Colab_Notebooks/bert_model2")
tokenizer.save_pretrained("/content/drive/MyDrive/Colab_Notebooks/bert_model2")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/Colab_Notebooks/bert_model2/tokenizer_config.json',
 '/content/drive/MyDrive/Colab_Notebooks/bert_model2/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = "/content/drive/MyDrive/Colab_Notebooks/bert_model2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# from datasets import Dataset
# import pandas as pd

# test_df = pd.DataFrame({
#     "text": X_test,
#     "label": y_test
# })

# test_dataset = Dataset.from_pandas(test_df)

In [ ]:
# from transformers import BertTokenizer

# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# def tokenize_function(example):
#     return tokenizer(
#         example["text"],
#         padding="max_length",
#         truncation=True,
#         max_length=128
#     )

# test_dataset = test_dataset.map(tokenize_function, batched=True)

# test_dataset.set_format(
#     type="torch",
#     columns=["input_ids", "attention_mask", "label"]
# )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/46415 [00:00<?, ? examples/s]

In [ ]:
# from transformers import Trainer

# trainer = Trainer(model=model)

In [ ]:
# predictions = trainer.predict(test_dataset)

# y_pred = predictions.predictions.argmax(axis=1)
# y_true = predictions.label_ids

In [ ]:
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# accuracy = accuracy_score(y_true, y_pred)
# precision = precision_score(y_true, y_pred)
# recall = recall_score(y_true, y_pred)
# f1 = f1_score(y_true, y_pred)

# cm = confusion_matrix(y_true, y_pred)

# print("Accuracy:", accuracy)
# print("Precision:", precision)
# print("Recall:", recall)
# print("F1 Score:", f1)
# print("Confusion Matrix:\n", cm)

Accuracy: 0.976516212431326
Precision: 0.9730909090909091
Recall: 0.9801353040031026
F1 Score: 0.9766004035893693
Confusion Matrix:
 [[22579   629]
 [  461 22746]]


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

label_map = {
    0: "Non Suicide",
    1: "Suicide"
}

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)

        pred_id = torch.argmax(probabilities, dim=1).item()

    print("=" * 80)
    print(f"Text: {text}")
    print(f"Prediction: {label_map[pred_id]}")
    print(f"Confidence: {probabilities[0][pred_id].item():.4f}")
    print(f"Probabilities: {probabilities[0].cpu().numpy()}")

    return label_map[pred_id], probabilities[0].cpu().numpy()

In [ ]:
test_cases = [
    # "I love my family.",
    # "Life is beautiful.",
    # "Hello.",
    # "Thanks.",
    # "I am very happy today.",
    # "I don't want to live anymore.",
    # "I love my parents.",
    # "Nobody understands me.",
    # "Hello how are you",
    # "I got a job today",
    # "Good Morning",
    # "not suicide"
    "Hii",
    "Thanks",
    "Lovely",
    "Genuine"
]

words = [

    # ---------- Clearly Non-Suicide ----------
    "Hi",
    "Hello",
    "Good morning.",
    "Thanks for your help.",
    "See you tomorrow.",
    "I had a great day today.",
    "Life is beautiful.",
    "I love my family.",
    "I love my parents.",
    "I love spending time with my friends.",
    "I enjoyed watching a movie today.",
    "The weather is amazing today.",
    "I cooked dinner for my family.",
    "I went for a morning walk.",
    "I am excited about my internship.",
    "I finally solved my coding problem.",
    "I am learning machine learning.",
    "I passed my exam.",
    "Today was productive.",
    "I feel relaxed after my workout.",

    # ---------- Neutral (Should be Non-Suicide) ----------
    "I bought groceries today.",
    "The meeting starts at 3 PM.",
    "My laptop battery is low.",
    "I need to charge my phone.",
    "I have a dentist appointment tomorrow.",
    "I drank coffee this morning.",
    "The train is running late.",
    "I finished reading my book.",
    "I answered all my emails.",
    "Today is Monday.",

    # ---------- Negative but NOT Suicide ----------
    "I failed my exam.",
    "I lost my wallet today.",
    "I'm feeling stressed about work.",
    "I'm exhausted after studying.",
    "I'm having a difficult week.",
    "I argued with my friend today.",
    "Nobody replied to my messages.",
    "I'm feeling lonely today.",
    "I'm disappointed with my performance.",
    "Things aren't going my way right now.",

    # ---------- Borderline ----------
    "I feel empty.",
    "I feel completely alone.",
    "I don't know what to do anymore.",
    "Nothing seems to make me happy.",
    "Everything feels overwhelming.",
    "I'm struggling to cope.",
    "I feel like a burden.",
    "I'm losing hope.",
    "I don't see much purpose lately.",
    "I wish things could change.",

    # ---------- Clear Suicide Risk ----------
    "I don't want to live anymore.",
    "I want to end my life.",
    "Everyone would be better off without me.",
    "I wish I could disappear forever.",
    "There is no reason for me to keep going.",
    "I can't take this pain anymore.",
    "I want everything to end.",
    "I have no reason to stay alive.",
    "I don't think I can continue living.",
    "Life isn't worth living anymore."
]

for text in words:
    predict(text)

Text: Hi
Prediction: Non Suicide
Confidence: 0.9761
Probabilities: [0.9760913  0.02390872]
Text: Hello
Prediction: Non Suicide
Confidence: 0.9861
Probabilities: [0.9861253  0.01387475]
Text: Good morning.
Prediction: Non Suicide
Confidence: 0.9809
Probabilities: [0.98085356 0.01914642]
Text: Thanks for your help.
Prediction: Non Suicide
Confidence: 0.9683
Probabilities: [0.96828353 0.03171646]
Text: See you tomorrow.
Prediction: Non Suicide
Confidence: 0.9511
Probabilities: [0.9511214  0.04887868]
Text: I had a great day today.
Prediction: Non Suicide
Confidence: 0.9985
Probabilities: [0.9985392  0.00146078]
Text: Life is beautiful.
Prediction: Non Suicide
Confidence: 0.9985
Probabilities: [0.9984712  0.00152877]
Text: I love my family.
Prediction: Non Suicide
Confidence: 0.9838
Probabilities: [0.9837675 0.0162325]
Text: I love my parents.
Prediction: Non Suicide
Confidence: 0.9948
Probabilities: [0.9948073  0.00519268]
Text: I love spending time with my friends.
Prediction: Non Suicid

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

test_loader = DataLoader(test_dataset, batch_size=32)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.9731
Precision: 0.9734
Recall   : 0.9724
F1 Score : 0.9729


In [ ]:
print(classification_report(
    all_labels,
    all_preds,
    target_names=["Non Suicide", "Suicide"]
))

              precision    recall  f1-score   support

 Non Suicide       0.97      0.97      0.97     11718
     Suicide       0.97      0.97      0.97     11603

    accuracy                           0.97     23321
   macro avg       0.97      0.97      0.97     23321
weighted avg       0.97      0.97      0.97     23321



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

print(cm)

[[11410   308]
 [  320 11283]]


In [ ]:
from sklearn.metrics import roc_auc_score

model.eval()

all_probs = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)[:, 1]
        all_probs.extend(probs.cpu().numpy())

roc_auc = roc_auc_score(all_labels, all_probs)

print(f"ROC-AUC Score: {roc_auc:.4f}")

ROC-AUC Score: 0.9955


In [ ]:
!pip install -q huggingface_hub

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/Colab_Notebooks/bert_model2",
    repo_id="ankitml01/mental-health-model",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/ankitml01/mental-health-model/commit/7b8a1cb0db55b928cc5ee06cb1d7b4603a3028ef', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7b8a1cb0db55b928cc5ee06cb1d7b4603a3028ef', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ankitml01/mental-health-model', endpoint='https://huggingface.co', repo_type='model', repo_id='ankitml01/mental-health-model'), pr_revision=None, pr_num=None)